# EMDGrid Colab Demo: EMD-L1 and Knothe-Rosenblatt Transport Plans

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tvercaut/emdgrid/blob/main/example/colab_demo.ipynb)

This notebook demonstrates how to compile and run **emdgrid** in Google Colab to compute exact EMD-L1 and Knothe-Rosenblatt (KR) optimal transport plans on 2D grid histograms, visualize their sparsity patterns, plot dense transport matrices, and display how mass from the source distribution is transported.

In [ ]:
# Install build dependencies and compile emdgrid in Google Colab if needed
import os
import sys

if 'google.colab' in sys.modules:
    !apt-get update -qq && apt-get install -y -qq cmake build-essential
    !git clone https://github.com/tvercaut/emdgrid.git
    %cd emdgrid
    !cmake -B build -DEMDGRID_BUILD_PYTHON_BINDINGS=ON
    !cmake --build build -j
    sys.path.insert(0, os.path.abspath('build/bindings/python'))
else:
    # Add local build path if running locally
    possible_paths = [
        os.path.abspath('build/bindings/python'),
        os.path.abspath('../build/bindings/python'),
    ]
    for p in possible_paths:
        if os.path.exists(p) and p not in sys.path:
            sys.path.insert(0, p)

import pyemdgrid

print(f"Successfully loaded pyemdgrid version: {pyemdgrid.version()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse

# Set random seed for reproducibility
np.random.seed(42)

## 1. Create 10x10 Grid Histograms

We generate two 10x10 normalized grid histograms ($H_1$ and $H_2$) representing probability distributions on a 2D integer grid.

In [ ]:
# Create a 10x10 spatial grid
grid_size = 10
x, y = np.meshgrid(np.arange(grid_size), np.arange(grid_size), indexing='ij')

# Define two 2D Gaussian distributions on the 10x10 grid
mu1, sigma1 = (2.5, 2.5), 1.5
mu2, sigma2 = (6.5, 6.5), 1.5

h1 = np.exp(-((x - mu1[0]) ** 2 + (y - mu1[1]) ** 2) / (2 * sigma1 ** 2))
h2 = np.exp(-((x - mu2[0]) ** 2 + (y - mu2[1]) ** 2) / (2 * sigma2 ** 2))

# Normalize histograms so total mass is 1.0
h1 /= h1.sum()
h2 /= h2.sum()

# Plot the source and target 10x10 histograms
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

im1 = axes[0].imshow(h1, cmap='viridis', origin='lower')
axes[0].set_title('Source Histogram $H_1$ (10x10)')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
fig.colorbar(im1, ax=axes[0], shrink=0.8)

im2 = axes[1].imshow(h2, cmap='viridis', origin='lower')
axes[1].set_title('Target Histogram $H_2$ (10x10)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
fig.colorbar(im2, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## 2. Compute Transport Plans with EMD-L1 and Knothe-Rosenblatt

We compute sparse transport plans for both solvers:
1. **EMD-L1**: Exact tree-based network simplex optimal transport solver under $L_1$ ground distance.
2. **Knothe-Rosenblatt (KR)**: Fast $N$-D heuristic optimal transport solver.

In [ ]:
# Compute EMD-L1 transport plan and cost
cost_emd, plan_emd = pyemdgrid.emd_l1(h1, h2, return_transport_plan=True)

# Compute Knothe-Rosenblatt transport plan and cost
cost_kr, plan_kr = pyemdgrid.knothe_rosenblatt(h1, h2, metric='l1', return_transport_plan=True)

print(f"EMD-L1 Cost : {cost_emd:.6f} | Non-zero entries (NNZ): {plan_emd.nnz}")
print(f"KR Cost     : {cost_kr:.6f} | Non-zero entries (NNZ): {plan_kr.nnz}")

## 3. Visualize Sparsity Patterns

The transport plans are returned as sparse matrices of shape (100, 100) representing the flow from the 100 flattened source cells to the 100 flattened target cells.
We plot their sparsity patterns using `plt.spy()`. 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

axes[0].spy(plan_emd, markersize=3, color='navy')
axes[0].set_title(f"EMD-L1 Transport Plan Sparsity Pattern\nCost: {cost_emd:.4f} | NNZ: {plan_emd.nnz}")
axes[0].set_xlabel("Target Cell Index (0..99)")
axes[0].set_ylabel("Source Cell Index (0..99)")

axes[1].spy(plan_kr, markersize=3, color='darkred')
axes[1].set_title(f"Knothe-Rosenblatt Transport Plan Sparsity Pattern\nCost: {cost_kr:.4f} | NNZ: {plan_kr.nnz}")
axes[1].set_xlabel("Target Cell Index (0..99)")
axes[1].set_ylabel("Source Cell Index (0..99)")

plt.tight_layout()
plt.show()

## 4. Visualize Dense Transport Plans with Fixed Colormap [0, 1]

Converting the transport plan matrices to dense arrays and displaying them as images using `imshow()` with a fixed colormap range `[0, 1]` (`vmin=0`, `vmax=1`).

In [ ]:
emd_dense = plan_emd.toarray()
kr_dense = plan_kr.toarray()

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

im0 = axes[0].imshow(emd_dense, cmap='viridis', vmin=0, vmax=1, origin='upper')
axes[0].set_title(f"EMD-L1 Dense Transport Matrix [0, 1]\nMax Value: {emd_dense.max():.4f}")
axes[0].set_xlabel("Target Cell Index (0..99)")
axes[0].set_ylabel("Source Cell Index (0..99)")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(kr_dense, cmap='viridis', vmin=0, vmax=1, origin='upper')
axes[1].set_title(f"Knothe-Rosenblatt Dense Transport Matrix [0, 1]\nMax Value: {kr_dense.max():.4f}")
axes[1].set_xlabel("Target Cell Index (0..99)")
axes[1].set_ylabel("Source Cell Index (0..99)")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## 5. Visualize Mass Transport / Displacement Vectors

To see how mass from $H_1$ is transported to $H_2$, we compute the average displacement vector (barycentric map) for each grid cell $i=(x,y)$:
$$\Delta x_i = \frac{\sum_j P_{ij} x_j}{h_{1,i}} - x_i, \quad \Delta y_i = \frac{\sum_j P_{ij} y_j}{h_{1,i}} - y_i$$
and overlay these transport displacement vectors on the source histogram $H_1$.

In [ ]:
# Flattened spatial grid coordinates for source and target cells
x_flat = x.ravel().astype(float)
y_flat = y.ravel().astype(float)
h1_flat = h1.ravel()

# Compute barycentric displacement vectors for source cells with positive mass
mask = h1_flat > 1e-12

def compute_displacement_field(plan_dense):
    target_x = np.zeros_like(h1_flat)
    target_y = np.zeros_like(h1_flat)
    target_x[mask] = (plan_dense[mask] @ x_flat) / h1_flat[mask]
    target_y[mask] = (plan_dense[mask] @ y_flat) / h1_flat[mask]
    dx = (target_x - x_flat).reshape(grid_size, grid_size)
    dy = (target_y - y_flat).reshape(grid_size, grid_size)
    return dx, dy

dx_emd, dy_emd = compute_displacement_field(emd_dense)
dx_kr, dy_kr = compute_displacement_field(kr_dense)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# EMD-L1 transport displacement field
im_emd = axes[0].imshow(h1, cmap='Blues', origin='lower', alpha=0.6)
axes[0].quiver(y, x, dy_emd, dx_emd, color='navy', angles='xy', scale_units='xy', scale=1, width=0.012)
axes[0].set_title("EMD-L1 Mass Transport Vectors")
axes[0].set_xlabel("X")
axes[0].set_ylabel("Y")

# Knothe-Rosenblatt transport displacement field
im_kr = axes[1].imshow(h1, cmap='Reds', origin='lower', alpha=0.6)
axes[1].quiver(y, x, dy_kr, dx_kr, color='darkred', angles='xy', scale_units='xy', scale=1, width=0.012)
axes[1].set_title("Knothe-Rosenblatt Mass Transport Vectors")
axes[1].set_xlabel("X")
axes[1].set_ylabel("Y")

plt.tight_layout()
plt.show()